# Unsupervised Learning: KMeans and Charlottesville Weather
**DS 6021 · Prince Afriyie · University of Virginia**

**Question:** Can a few representative weather profiles summarize the variety of weather hours near Charlottesville?

We have measurements, but no response variable $Y$ to predict. We will explore the features, fit KMeans, inspect the groups, and critique what the groups mean.

### Learning goals
- Explain the difference between KNN and KMeans.
- Connect nearest-center assignment and mean updates to the squared-error objective.
- Fit and interpret KMeans with simple Python and Plotly.
- Explain why feature choice, scaling, initialization, and $K$ matter.
- Use inertia and silhouette as diagnostic evidence rather than proof of true classes.

[Open the convergence dashboard](https://afriyiep1.github.io/ds6021-predictive-modeling/kmeans/)

The main examples use a **fixed real-data snapshot** for reproducibility. An optional cell at the end retrieves recent weather. No train/test split is needed for this descriptive exercise. If our aim later becomes evaluating performance on future weather, we would design an appropriate temporal evaluation.


## 1. What does one row represent?

One row is one hour at a model grid point near Charlottesville (requested coordinates: 38.0293, −78.4767). The snapshot contains **1,440 hours**, from **September 11, 2026, 00:00 UTC through November 9, 2026, 23:00 UTC**, fetched on November 10, 2026.

These are **Open-Meteo model-based hourly estimates from recent model runs**, not direct station observations. All hours, including nighttime, are retained.

| Variable | Meaning | Use |
|---|---|---|
| `cloud_cover` | Total cloud cover, % | 2D and 3D clustering |
| `relative_humidity_2m` | Relative humidity at 2 m, % | 2D and 3D clustering |
| `precipitation` | Total precipitation in the preceding hour, mm; includes rain, showers, and snow water equivalent | Added in 3D |
| `temperature_2m` | Temperature at 2 m, °C | Context only |
| `weather_code` | Provider's WMO weather code | Context only; never a clustering feature |
| `time` | Unix timestamp in UTC | Hover labels and time ordering |

Low cloud cover at night is **clear**, not necessarily **sunny**. We do not tell the algorithm to produce rainy/cloudy/sunny labels.

Source: [Open-Meteo documentation](https://open-meteo.com/en/docs). Weather data by [Open-Meteo.com](https://open-meteo.com/), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/); reformatted and clustered for teaching.


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score

url = "https://afriyiep1.github.io/ds6021-predictive-modeling/kmeans/weather-snapshot.csv"
weather = pd.read_csv(url)

weather["time_utc"] = pd.to_datetime(weather["time"], unit="s", utc=True)
weather["time_local"] = weather["time_utc"].dt.tz_convert("America/New_York")
weather["hour_label"] = weather["time_local"].dt.strftime("%Y-%m-%d %H:%M %Z")

features_2d = ["cloud_cover", "relative_humidity_2m"]
features_3d = features_2d + ["precipitation"]

print("Hourly records:", len(weather))
print("UTC range:", weather["time_utc"].min(), "to", weather["time_utc"].max())
display(weather[["hour_label"] + features_3d].head())
display(weather[features_3d].isna().sum().rename("Missing values"))


## 2. Look before fitting

Each point is an hour. Some points overlap because multiple hours have the same rounded measurements. Imagine placing three representative points on this plot. Which hours would each representative summarize?


In [ ]:
fig = px.scatter(
    weather, x="cloud_cover", y="relative_humidity_2m",
    hover_name="hour_label", hover_data=["precipitation", "temperature_2m"],
    opacity=0.45,
    labels={"cloud_cover": "Cloud cover (%)", "relative_humidity_2m": "Relative humidity (%)"},
    title="Charlottesville weather hours: no cluster labels yet"
)
fig.update_traces(marker_color="#232D4B", marker_size=6)
fig.show()


## 3. Fit a two-feature KMeans model

KMeans minimizes total squared distance to assigned centers:

$$J = \sum_{i=1}^{n}\|\mathbf z_i-\boldsymbol\mu_{c_i}\|^2.$$

Here $\mathbf z_i$ contains the standardized measurements of hour $i$, $c_i$ is its cluster ID, and $\boldsymbol\mu_{c_i}$ is that cluster's center.

`StandardScaler` gives each included feature mean zero and population standard deviation one. This is a modeling choice about relative feature weight, not an automatic guarantee of useful groups.

We use `n_init=20` to try 20 starts and keep the lowest inertia. `random_state=6021` makes this reproducible. `tol=0` is used here so the fitted centers match the final group means at a stable solution; `max_iter` still places a limit on the iterations.


In [ ]:
X2 = weather[features_2d].dropna().copy()

scaler2 = StandardScaler()
Z2 = scaler2.fit_transform(X2)

model2 = KMeans(
    n_clusters=3, init="k-means++", n_init=20,
    random_state=6021, algorithm="lloyd", tol=0, max_iter=300
)
labels2 = model2.fit_predict(Z2)

weather2 = weather.loc[X2.index].copy()
weather2["cluster"] = labels2.astype(str)

print("2D inertia:", round(model2.inertia_, 3))
print("Iterations in the selected run:", model2.n_iter_)


## 4. Inspect the groups in original units

Cluster IDs are arbitrary names, not ordered categories. Inspect sizes and feature values before choosing descriptive names. Temperature and precipitation below summarize the fitted groups; they were not inputs to this 2D model.


In [ ]:
profile2 = weather2.groupby("cluster").agg(
    hours=("time", "size"),
    cloud_cover_pct=("cloud_cover", "mean"),
    humidity_pct=("relative_humidity_2m", "mean"),
    precipitation_mm=("precipitation", "mean"),
    temperature_C=("temperature_2m", "mean")
)
display(profile2.round(2))

centers2 = pd.DataFrame(
    scaler2.inverse_transform(model2.cluster_centers_), columns=features_2d
)
centers2["cluster"] = centers2.index.astype(str)

fig = px.scatter(
    weather2, x="cloud_cover", y="relative_humidity_2m", color="cluster",
    hover_name="hour_label", hover_data=["precipitation"], opacity=0.50,
    color_discrete_sequence=["#276582", "#D16A25", "#5E478C"],
    labels={"cloud_cover": "Cloud cover (%)", "relative_humidity_2m": "Relative humidity (%)"},
    title="2D KMeans: fitted in standardized space, shown in original units"
)
fig.add_trace(go.Scatter(
    x=centers2["cloud_cover"], y=centers2["relative_humidity_2m"],
    mode="markers+text", text=centers2["cluster"], textposition="top center",
    marker=dict(size=18, symbol="x", color="black"), name="Centroids"
))
fig.show()


**Interpretation for this snapshot:** the 2D model separates clearer/humid, clearer/drier, and cloudy/humid hours. Humidity alone does not imply rain. Because rainfall is not an input here, do not describe this model as a rain detector.

The plot uses original units for interpretation. The dashboard plots the standardized coordinates when standardization is on, so displayed geometry matches the distances being optimized.


## 5. Add precipitation: a new three-feature fit

Adding a coordinate changes the distance calculation and potentially every cluster assignment. This is a new model, not simply a 3D view of the preceding 2D model.


In [ ]:
X3 = weather[features_3d].dropna().copy()

scaler3 = StandardScaler()
Z3 = scaler3.fit_transform(X3)

model3 = KMeans(
    n_clusters=3, init="k-means++", n_init=20,
    random_state=6021, algorithm="lloyd", tol=0, max_iter=300
)
labels3 = model3.fit_predict(Z3)

weather3 = weather.loc[X3.index].copy()
weather3["cluster"] = labels3.astype(str)
weather3["wet_hour"] = weather3["precipitation"] > 0

profile3 = weather3.groupby("cluster").agg(
    hours=("time", "size"),
    cloud_cover_pct=("cloud_cover", "mean"),
    humidity_pct=("relative_humidity_2m", "mean"),
    precipitation_mm=("precipitation", "mean"),
    wet_hours=("wet_hour", "sum")
)
display(profile3.round(3))
print("3D inertia:", round(model3.inertia_, 3))
print("3D silhouette:", round(silhouette_score(Z3, labels3), 3))


In [ ]:
plot3 = pd.DataFrame(Z3, columns=["Cloud cover (z)", "Humidity (z)", "Precipitation (z)"])
plot3["cluster"] = labels3.astype(str)
plot3["hour"] = weather3["hour_label"].to_numpy()

fig = px.scatter_3d(
    plot3, x="Cloud cover (z)", y="Humidity (z)", z="Precipitation (z)",
    color="cluster", hover_name="hour", opacity=0.6,
    color_discrete_sequence=["#276582", "#D16A25", "#5E478C"],
    title="3D KMeans in the standardized coordinates used for fitting"
)
fig.update_traces(marker_size=3)
fig.add_trace(go.Scatter3d(
    x=model3.cluster_centers_[:, 0], y=model3.cluster_centers_[:, 1],
    z=model3.cluster_centers_[:, 2], mode="markers",
    marker=dict(size=8, symbol="diamond", color="black"), name="Centroids"
))
fig.update_layout(scene=dict(aspectmode="data"), height=650)
fig.show()


**What should we say about these clusters?**

For this snapshot and fit, the group sizes are **954, 474, and 12 hours** (the numerical IDs may be permuted). They describe mostly clearer/dry hours, cloudy/humid hours with some light precipitation, and a small group of wetter hours. **32 wet hours remain in the large cloudy group**, while 12 form the wetter group. A cluster is therefore not equivalent to a weather-service class.

Precipitation is mostly zero and right-skewed. Standardization gives it comparable overall variance, but a few large values can still have strong influence. Try `np.log1p` on precipitation as a sensitivity analysis only after discussing how that changes the meaning of distance.

Do not compare the numerical 2D and 3D inertia values as if they measure the same problem. They sum errors across different feature spaces.


## 6. How many clusters?

Use the **2D standardized cloud-cover/humidity data**, matching the elbow and silhouette plots in the lecture. Keep the observations, features, and scaling fixed while varying only $K$. To investigate the 3D alternative afterward, change both occurrences of `Z2` in the next cell to `Z3`.

- **Inertia / WCSS:** lower means a more compact fit. The best attainable inertia cannot increase when $K$ increases, so choosing the smallest inertia by itself would encourage too many clusters. Fitted local solutions can occasionally deviate from this pattern.
- **Silhouette:** compares an observation's average distance to other points in its own cluster with its lowest average distance to a different cluster. It is not classification accuracy. It is defined only when there are at least two clusters and fewer clusters than observations.

$$s(i)=\frac{b(i)-a(i)}{\max\{a(i),b(i)\}}.$$

A value near 1 suggests strong separation, near 0 suggests overlap, and negative values suggest a point is closer on average to another cluster under the chosen distance.


In [ ]:
results = []

for k in range(1, 9):
    model = KMeans(
        n_clusters=k, init="k-means++", n_init=20,
        random_state=6021, algorithm="lloyd", tol=0, max_iter=300
    )
    labels = model.fit_predict(Z2)

    silhouette = np.nan
    if k > 1:
        silhouette = silhouette_score(Z2, labels)

    results.append({"K": k, "Inertia": model.inertia_, "Silhouette": silhouette})

diagnostics = pd.DataFrame(results)
display(diagnostics.round(3))

px.line(diagnostics, x="K", y="Inertia", markers=True,
        title="Elbow diagnostic: the same standardized 2D dataset").show()
px.line(diagnostics, x="K", y="Silhouette", markers=True,
        title="Silhouette diagnostic: higher is better separated, not necessarily more useful").show()


**Discuss:** Is there a decisive elbow? Are silhouette differences large enough to settle the choice? Is the 12-hour group scientifically useful or mostly a consequence of squared distance? A sensible choice of K combines these diagnostics, stability, and the purpose of the analysis. KMeans will return groups even when the data do not contain distinct natural categories.


## 7. Why do the starting centers matter?

Keep the data and K fixed. Use a single random start for each seed so that variability across starts is visible. The dashboard also shows one run at a time. Its JavaScript random number generator and standard KMeans++ initialization are not the same as scikit-learn's, so the same seed number does not imply the same starting points across the two tools.


In [ ]:
starts = []

for seed in range(12):
    model = KMeans(
        n_clusters=3, init="random", n_init=1,
        random_state=seed, algorithm="lloyd", tol=0, max_iter=300
    )
    model.fit(Z3)
    starts.append({"Seed": seed, "Inertia": model.inertia_, "Iterations": model.n_iter_})

starts = pd.DataFrame(starts)
display(starts.round(3))
px.bar(starts, x="Seed", y="Inertia", hover_data=["Iterations"],
       title="Same weather, same K, different random starts").show()


## 8. What changes without standardization?

Fit on the original percentages and millimeters. Compare the profiles and groupings, not the inertia numbers: those numbers use different scales. A numerical cluster ID can change even when a partition is identical, so comparing `labels_raw == labels3` is misleading.

The adjusted Rand index (ARI), used here only as an optional comparison, ignores the arbitrary cluster ID names. ARI = 1 means identical partitions up to relabeling. It does not establish that either partition is scientifically correct.


In [ ]:
model_raw = KMeans(
    n_clusters=3, init="k-means++", n_init=20,
    random_state=6021, algorithm="lloyd", tol=0, max_iter=300
)
labels_raw = model_raw.fit_predict(X3)

comparison = weather3.copy()
comparison["raw_cluster"] = labels_raw.astype(str)
display(comparison.groupby("raw_cluster")[features_3d].mean().round(3))
print("Agreement between raw and standardized partitions (ARI):",
      round(adjusted_rand_score(labels3, labels_raw), 3))


## 9. See one assignment and one update directly (optional)

This section exposes the mechanics on the **same 2D weather data**. It is not needed to use scikit-learn. We choose three actual hours as initial centers, then calculate all distances.

During **assignment**, centers stay fixed while labels may change. During **update**, labels stay fixed while each nonempty cluster's center moves to its coordinatewise mean.


In [ ]:
centers = Z2[[0, 480, 960]].copy()

# One column of squared distances for each center.
distances = np.column_stack([
    ((Z2 - center) ** 2).sum(axis=1) for center in centers
])
labels = distances.argmin(axis=1)
sse_assignment = ((Z2 - centers[labels]) ** 2).sum()

display(pd.DataFrame(distances[:6], columns=["Center 0", "Center 1", "Center 2"]).round(3))

new_centers = centers.copy()
for k in range(3):
    members = Z2[labels == k]
    if len(members) > 0:
        new_centers[k] = members.mean(axis=0)
    # If a cluster is empty, retain its previous center for this illustration.

sse_update = ((Z2 - new_centers[labels]) ** 2).sum()
print("SSE after assignment:", round(sse_assignment, 3))
print("SSE after mean update:", round(sse_update, 3))
print("An exact mean update cannot increase SSE:", sse_update <= sse_assignment + 1e-9)


Continue these steps in the dashboard. Reset with the same settings to reproduce a run; change the seed to change the starting centers; switch from 2D to 3D to fit a different feature space. The history slider moves between recorded algorithm states, and the SSE chart follows those same states.

**Explain:** Why can the objective stay unchanged? Why does a stable assignment fail to guarantee the globally best partition? Why should the centers not move during an assignment step?


## 10. Refresh recent data (optional)

The snapshot above remains fixed. Set the flag to `True` to retrieve the most recent **60 completed UTC days** using the same variables. The API is public, and the call requires internet access. The resulting `recent_weather` is a separate DataFrame; replacing the snapshot and rerunning the fitting cells will change the results.

Unix UTC timestamps are converted to America/New_York locally, which handles daylight-saving changes correctly. The API's WMO weather code stays out of the clustering matrix.


In [ ]:
RUN_LIVE_REFRESH = False

if RUN_LIVE_REFRESH:
    import json
    from urllib.request import urlopen
    from datetime import datetime, timezone

    api_url = (
        "https://api.open-meteo.com/v1/forecast?latitude=38.0293&longitude=-78.4767"
        "&hourly=temperature_2m,relative_humidity_2m,precipitation,cloud_cover,weather_code"
        "&timezone=UTC&timeformat=unixtime&past_days=60&forecast_days=0"
    )
    with urlopen(api_url, timeout=30) as response:
        payload = json.load(response)

    recent_weather = pd.DataFrame(payload["hourly"])
    recent_weather["time_utc"] = pd.to_datetime(recent_weather["time"], unit="s", utc=True)
    recent_weather["time_local"] = recent_weather["time_utc"].dt.tz_convert("America/New_York")
    recent_weather["hour_label"] = recent_weather["time_local"].dt.strftime("%Y-%m-%d %H:%M %Z")

    print("Fetched:", datetime.now(timezone.utc).isoformat())
    print("Model-based estimates; not station observations.")
    display(recent_weather.head())
else:
    print("Using the reproducible saved snapshot. Set RUN_LIVE_REFRESH=True to retrieve recent estimates.")


## Interpretation checklist

1. What exactly is one point, and what features define distance?
2. How does scaling change the scientific meaning of similarity?
3. Are the groups compact, stable, and useful for the question?
4. Do initializations produce different final solutions?
5. Does a group represent a useful pattern, a rare extreme, or an arbitrary split of a continuum?
6. Can a proposed group name describe all its members reasonably well?

## Readings and references

**Core:** *An Introduction to Statistical Learning, with Applications in Python*, Chapter 12: the challenge of unsupervised learning, KMeans clustering, and practical issues in clustering. [Official book and resources](https://www.statlearning.com/).

**Implementation:** [scikit-learn KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html), [clustering user guide](https://scikit-learn.org/stable/modules/clustering.html#k-means), and [silhouette examples](https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_silhouette_analysis.html).

**Data:** [Open-Meteo Forecast API](https://open-meteo.com/en/docs). Weather data by Open-Meteo.com under CC BY 4.0; reformatted and clustered for teaching. A raw provider response and provenance metadata accompany the dashboard's saved snapshot.
